In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS asb.semantic
;

In [0]:
%sql
-- ----------------------------------------------------------------------------
-- ORDER HEADER GRAIN: Delivery, Logistics and Customer Retention
-- ----------------------------------------------------------------------------
CREATE OR REPLACE VIEW asb.semantic.delivery_logistics AS
SELECT 
    -- Applicable columns for global filter
    o.purchase_date_key,
    d.year,
    d.quarter,
    d.year_quarter,
    d.month,
    d.month_name,
    d.year_month,
    o.order_status,
    c.state AS customer_state_code,
    CASE c.state
        WHEN 'SP' THEN 'São Paulo'
        WHEN 'RJ' THEN 'Rio de Janeiro'
        WHEN 'MG' THEN 'Minas Gerais'
        WHEN 'RS' THEN 'Rio Grande do Sul'
        WHEN 'PR' THEN 'Paraná'
        WHEN 'SC' THEN 'Santa Catarina'
        WHEN 'BA' THEN 'Bahia'
        WHEN 'DF' THEN 'Distrito Federal'
        WHEN 'ES' THEN 'Espírito Santo'
        WHEN 'GO' THEN 'Goiás'
        WHEN 'PE' THEN 'Pernambuco'
        WHEN 'CE' THEN 'Ceará'
        WHEN 'PA' THEN 'Pará'
        WHEN 'MT' THEN 'Mato Grosso'
        WHEN 'MA' THEN 'Maranhão'
        WHEN 'MS' THEN 'Mato Grosso do Sul'
        WHEN 'PB' THEN 'Paraíba'
        WHEN 'PI' THEN 'Piauí'
        WHEN 'RN' THEN 'Rio Grande do Norte'
        WHEN 'AL' THEN 'Alagoas'
        WHEN 'SE' THEN 'Sergipe'
        WHEN 'TO' THEN 'Tocantins'
        WHEN 'RO' THEN 'Rondônia'
        WHEN 'AM' THEN 'Amazonas'
        WHEN 'AC' THEN 'Acre'
        WHEN 'AP' THEN 'Amapá'
        WHEN 'RR' THEN 'Roraima'
        ELSE c.state
    END AS customer_state,
    -- Entity and metric columns
    o.order_id,
    c.customer_id,
    c.customer_unique_id,
    c.customer_type,
    o.delivery_duration_days,
    o.estimated_delivery_duration_days,
    o.is_delivered,
    o.is_late_delivery
FROM asb.olist_gold.fact_orders o
LEFT JOIN asb.olist_gold.dim_customers c 
    ON o.customer_id = c.customer_id
LEFT JOIN asb.olist_gold.dim_date d 
    ON o.purchase_date_key = d.date_key
;

In [0]:
%sql
-- ----------------------------------------------------------------------------
-- LINE-ITEM GRAIN: Revenue, Unit Economics, Products and Sellers
-- ----------------------------------------------------------------------------
CREATE OR REPLACE VIEW asb.semantic.order_item_sales AS
SELECT 
    -- Applicable columns for global filter
    o.purchase_date_key,
    d.year,
    d.quarter,
    d.year_quarter,
    d.month,
    d.month_name,
    d.year_month,
    o.order_status,
    c.state AS customer_state,
    -- Entity and metric columns
    i.order_id,
    i.order_item_id,
    c.customer_unique_id,
    i.product_id,
    p.product_category,
    i.seller_id,
    s.state AS seller_state,
    s.city AS seller_city,
    i.price AS item_price,
    i.freight_value,
    i.gross_item_revenue
FROM asb.olist_gold.fact_order_items i
JOIN asb.olist_gold.fact_orders o 
    ON i.order_id = o.order_id
LEFT JOIN asb.olist_gold.dim_products p 
    ON i.product_id = p.product_id
LEFT JOIN asb.olist_gold.dim_sellers s 
    ON i.seller_id = s.seller_id
LEFT JOIN asb.olist_gold.dim_customers c 
    ON o.customer_id = c.customer_id
LEFT JOIN asb.olist_gold.dim_date d 
    ON o.purchase_date_key = d.date_key
;

In [0]:
%sql
-- ----------------------------------------------------------------------------
-- FEEDBACK GRAIN: Customer Reviews and Satisfaction
-- ----------------------------------------------------------------------------
CREATE OR REPLACE VIEW asb.semantic.order_satisfaction AS
SELECT 
    -- Applicable columns for global filter
    o.purchase_date_key,
    d.year,
    d.quarter,
    d.year_quarter,
    d.month,
    d.month_name,
    d.year_month,
    o.order_status,
    c.state AS customer_state,
    -- Entity and metric columns
    r.review_id,
    r.order_id,
    r.review_score,
    o.is_late_delivery,
    o.delivery_duration_days
FROM asb.olist_gold.fact_order_reviews r
JOIN asb.olist_gold.fact_orders o 
    ON r.order_id = o.order_id
LEFT JOIN asb.olist_gold.dim_customers c 
    ON o.customer_id = c.customer_id
LEFT JOIN asb.olist_gold.dim_date d 
    ON o.purchase_date_key = d.date_key
;